# Practical 3 — Bank Customer Churn Prediction (Neural Network)
**Aim:** Predict if a bank customer leaves (`Exited=1`) or stays (`Exited=0`). Compare basic and improved neural networks.

**Dataset:** `Churn_Modelling.csv` — 10,000 customer records, 14 columns. Keep it in the same folder as this notebook.

**Instructions:** Run each code cell in order using **Shift+Enter**. Read the Markdown explanation above each code cell.

## 1. What are we predicting?
**Churn** = a customer leaving the bank. This is **binary classification** because the answer is 0 (stays) or 1 (leaves).

A **neural network** learns patterns by adjusting weights between layers of neurons.

**Memory trick:** Customer information → Neural network → Stay (0) / Leave (1).

## 2. Install libraries (only once)
In VS Code terminal run:

`python -m pip install pandas numpy scikit-learn tensorflow ipykernel`

If an import fails, ensure VS Code has selected the same Python environment where packages were installed.

## 3. Imports
- **pandas** = tables/CSV; **numpy** = numbers; **TensorFlow/Keras** = neural networks.
- **train_test_split** = divide into learning and testing data.
- **StandardScaler** = scale values; **accuracy_score** = proportion correct; **confusion_matrix** = correct/incorrect predictions.
- **Sequential** = layers in order; **Dense** = neurons; **Dropout** = randomly disable some neurons while training.

In [ ]:
import pandas as pd              # Work with CSV tables
import numpy as np               # Math
import tensorflow as tf          # Neural networks
from sklearn.model_selection import train_test_split  # Split dataset
from sklearn.preprocessing import StandardScaler       # Scale inputs
from sklearn.metrics import accuracy_score, confusion_matrix  # Evaluate
from tensorflow.keras.models import Sequential  # Create network
from tensorflow.keras.layers import Input, Dense, Dropout  # Layers

np.random.seed(42)               # More repeatable results
tf.random.set_seed(42)


## 4. Read the CSV
`read_csv()` loads a table; `head()` displays the first 5 records; `shape` shows rows and columns; `isnull().sum()` checks missing values.

Important columns: `CreditScore`, `Geography`, `Gender`, `Age`, `Balance`, `IsActiveMember`, and `Exited` (our answer).

In [ ]:
df = pd.read_csv("Churn_Modelling.csv")  # Open file
print("Shape:", df.shape)          # Expected (10000, 14)
print(df.head())                    # First 5 customers
print(df.isnull().sum())            # Missing values per column


## 5. Preprocessing: remove IDs, convert text, separate X and y
`RowNumber`, `CustomerId`, `Surname` are identifiers; remove them for this simple model.

`get_dummies()` converts categories (Geography, Gender) into 0/1 columns, because the neural network needs numeric inputs.

**X = input customer details. y = output Exited (0 or 1).**

In [ ]:
df = df.drop(columns=["RowNumber", "CustomerId", "Surname"])  # Remove IDs
df = pd.get_dummies(df, columns=["Geography", "Gender"], drop_first=True)  # Text to numbers
X = df.drop(columns=["Exited"]).astype("float32")  # Inputs
y = df["Exited"]                                 # Answer
print("Number of features:", X.shape[1])


## 6. Train-test split
We train on **80%** and test on the remaining **20%**.

`test_size=0.2` = 20% test. `random_state=42` = repeatable split. `stratify=y` keeps roughly the same proportion of customers who leave in each part.

**Memory:** Train = practice; Test = exam.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)


## 7. StandardScaler: make features comparable
A balance might be 100000, but age is 40. Scaling puts features on comparable scales.

`fit_transform(X_train)` learns mean and standard deviation **from training data** and scales it. `transform(X_test)` uses those same values for test data; never fit again on the test data.

`shape[1]` = number of input columns.

In [ ]:
scaler = StandardScaler()                  # Create scaler
X_train = scaler.fit_transform(X_train)    # Learn scaling and apply to train
X_test = scaler.transform(X_test)          # Apply same scaling to test
n_features = X_train.shape[1]             # Number of inputs
print("Input features:", n_features)


## 8. Neural network words to know
- **Sequential** = layers stacked one after another.
- **Input** = number of input features.
- **Dense(16)** = a fully connected layer with 16 neurons.
- **ReLU** = activation used inside the network to learn complex patterns.
- **Sigmoid** = output between 0 and 1, treated as estimated chance of churn.
- **Adam** = optimizer that adjusts model weights.
- **Binary crossentropy** = loss function for two classes.
- **Epoch** = one full pass through training data.
- **Batch size 32** = process 32 examples per training step.
- **fit** = train; **predict** = get predictions.

For example: output 0.8 → predict **1 (leave)** using a 0.5 threshold.

## 9. Reusable function: train and evaluate
This avoids writing the same training code twice.

We use **20 epochs** for quicker exam practice. The supplied code uses **50 epochs**; change `epochs=20` to `epochs=50` if your examiner requires it. More epochs are not automatically better.

`verbose=0` hides long training logs. `astype(int)` turns True/False into 1/0.

In [ ]:
def train_and_test(model):
    # Tell the network how to learn and what metric to show
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

    # Train model for 20 rounds using batches of 32 customers
    model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)

    # Get probabilities; above 0.5 means customer predicted to leave
    pred = (model.predict(X_test, verbose=0) > 0.5).astype(int)

    # Print results and return accuracy for comparison
    acc = accuracy_score(y_test, pred)
    print("Accuracy:", acc)
    print("Confusion matrix:")
    print(confusion_matrix(y_test, pred))
    return acc


## 10. Basic model
Structure: **Input → 16 neurons (ReLU) → 8 neurons (ReLU) → 1 output (Sigmoid)**.

The output neuron predicts a probability between 0 and 1.

In [ ]:
basic = Sequential([
    Input(shape=(n_features,)),       # Number of input columns
    Dense(16, activation="relu"),    # Hidden layer 1: 16 neurons
    Dense(8, activation="relu"),     # Hidden layer 2: 8 neurons
    Dense(1, activation="sigmoid")   # Output: probability of leaving
])
print("BASIC MODEL")
basic_acc = train_and_test(basic)


## 11. Improved model: more neurons + Dropout
Structure: **Input → 32 neurons → Dropout(0.2) → 16 neurons → Dropout(0.2) → 1 output**.

`Dropout(0.2)` randomly disables about 20% of the layer's outputs during training. This helps reduce **overfitting** (memorizing training data but doing poorly on new data).

**Important:** A bigger model is not guaranteed to have higher test accuracy.

In [ ]:
improved = Sequential([
    Input(shape=(n_features,)),       # Same input size
    Dense(32, activation="relu"),    # More neurons
    Dropout(0.2),                    # Turn off ~20% during training
    Dense(16, activation="relu"),    # Second hidden layer
    Dropout(0.2),                    # Another dropout layer
    Dense(1, activation="sigmoid")   # Output 0 to 1
])
print("IMPROVED MODEL")
improved_acc = train_and_test(improved)


## 12. Compare the two models
Higher **test accuracy** means more test customers were classified correctly. It is possible for the improved model to do worse.

In [ ]:
print("Basic model accuracy   :", round(basic_acc, 4))
print("Improved model accuracy:", round(improved_acc, 4))
if improved_acc > basic_acc:
    print("Improved model has higher test accuracy")
elif improved_acc < basic_acc:
    print("Basic model has higher test accuracy")
else:
    print("Both have the same test accuracy")


## 13. Confusion matrix: understand the output

| Actual / Predicted | Predicted stay (0) | Predicted leave (1) |
|---|---|---|
| **Actually stayed (0)** | TN: correct stay | FP: wrong leave |
| **Actually left (1)** | FN: wrong stay | TP: correct leave |

**Accuracy = (TP + TN) / total predictions.**

A **false negative** is a customer who actually leaves but was predicted to stay — important for a bank trying to retain customers.

## 14. Viva questions — short answers

1. **What is churn?** Customer leaving a bank or service.
2. **What is the target?** `Exited`, where 0 = stays and 1 = leaves.
3. **What is binary classification?** Predicting one of two categories.
4. **What is a neural network?** Layers of connected units that learn from data.
5. **What is a neuron?** A computation combining inputs and an activation function.
6. **What is Sequential?** Keras model that stacks layers in order.
7. **What is Dense?** Fully connected layer of neurons.
8. **What is ReLU?** Returns 0 for negative inputs, input itself for positive inputs.
9. **What is Sigmoid?** Converts a number into a value between 0 and 1.
10. **Why sigmoid at output?** We need a probability for two-class classification.
11. **What is an epoch?** One full pass through training examples.
12. **What is batch size?** Number of examples used in one training step.
13. **What is Adam?** An optimizer for updating weights.
14. **What is loss?** A numerical measure of how wrong predictions are during training.
15. **Why binary crossentropy?** A standard loss for binary classification.
16. **What is Dropout?** Randomly disables outputs during training to reduce overfitting.
17. **What is overfitting?** Good training performance but poor new-data performance.
18. **Why scale features?** To make training easier when feature magnitudes differ.
19. **What is one-hot encoding?** Convert text categories into numeric indicator columns.
20. **What is a confusion matrix?** Table showing correct and wrong predictions by class.
21. **What is accuracy?** Correct predictions / all predictions.
22. **Is a larger model always better?** No; compare performance on test data.

## 15. Last-minute memory trick

**Load → Drop IDs → Dummies → X/y → Split → Scale → Basic → Improved → Compare**

`read_csv` = load; `get_dummies` = convert text; `train_test_split` = split; `StandardScaler` = scale; `Sequential` = model; `Dense` = neurons; `Dropout` = prevent overfitting; `compile` = configure; `fit` = train; `predict` = predict.

**Exam tip:** Run all cells in order. Do not skip the preprocessing and scaling cells.